# Review information: full-text criteria instead of the LGAR criteria

Follow-up to the pattern analysis: part of the gain of the review abstract seemed to come from the LGAR criteria being condensed, sometimes with omissions (e.g. CD012069 leaves out the RCT arms the review included).

**Review information = only the review's own section *Criteria for considering studies for this review***, taken as written from the Cochrane full text (see [04_fulltext_criteria_extraction.ipynb](04_fulltext_criteria_extraction.ipynb)). No review title, no research questions. Records, record input, question and model are unchanged.

| | `state.review` |
|---|---|
| baseline | title, research questions, inclusion criteria, exclusion criteria (LGAR) |
| review abstract runs | title, review abstract (full, or without Main results / Conclusions) |
| earlier variant (`fulltext_criteria_rq`) | title, research questions (LGAR), criteria (full text) |
| this run | criteria (full text) |

Note: this run also drops the review title, which every other configuration has.

Caveats: the full-text criteria are 1.5–12k characters, longer than the LGAR criteria. A few contain notes written after screening (references to *Differences between protocol and review* in CD012069 and CD012164, summary-of-findings tables in 4 reviews); these are about reporting outcomes, not about which studies were included.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = True    # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("review_fulltext_criteria_only")
COMPARE = {"baseline": S.out_dir("baseline"),
           "title_abstract": S.out_dir("review_title_abstract"),
           "no_results": S.out_dir("review_title_abstract_no_results"),
           "fulltext_criteria_rq": S.out_dir("review_fulltext_criteria"),
           "fulltext_criteria": OUT}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(FT) == set(CRITERIA), f"full-text criteria missing for {set(CRITERIA) - set(FT)}"

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({"lgar_criteria_chars": {r: len(c["inclusion_criteria"]) + len(c["exclusion_criteria"]) for r, c in CRITERIA.items()},
              "fulltext_criteria_chars": {r: len(v["criteria"]) for r, v in FT.items()},
              "doi": {r: v["doi"] for r, v in FT.items()},
              "records": n, "in_subset": {r: r in SUBSET for r in n.index}})

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


,lgar_criteria_chars,fulltext_criteria_chars,doi,records,in_subset
CD000996,406,1463,10.1002/14651858.CD000996.pub3,281,True
CD001261,753,2166,10.1002/14651858.CD001261.pub4,571,True
CD004414,1331,3241,10.1002/14651858.CD004414.pub3,336,True
CD006468,779,1775,10.1002/14651858.CD006468.pub6,3874,False
CD007867,766,3423,10.1002/14651858.CD007867.pub2,943,True
CD009069,1493,3834,10.1002/14651858.CD009069.pub3,1757,False
CD009642,1030,3958,10.1002/14651858.CD009642.pub3,1922,False
CD010038,1381,12192,10.1002/14651858.CD010038.pub3,8867,False
CD010239,616,2599,10.1002/14651858.CD010239.pub2,224,True
CD010558,1513,6264,10.1002/14651858.CD010558.pub2,2815,False


## Decision model input

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {
            "review": {
                "criteria": FT[review]["criteria"],
            },
            "record": {"title": title, "abstract": abstract},
        },
        "questions": S.QUESTIONS,
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:3000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "criteria": "### Criteria for considering studies for this review\n#### Types of studies\nAll randomised controlled trials (RCTs) using inhaled corticosteroids (ICS) in patients with bronchiectasis, in comparison to placebo.\n#### Types of participants\nChildren or adults with bronchiectasis (defined clinically or radiologically) not related to cystic fibrosis.\n#### Types of interventions\nAll types of ICS.\n#### Types of outcome measures\n##### Primary outcomes\nChange in objective measures of lung function.\n##### Secondary outcomes\n###### (A) for short\u2010term effectiveness (6 months or less)\nMean difference in bronchiectasis severity control (wheeze, dyspnoea, cough diary, etc).\nMean of respiratory exacerbations, or hospitalisations per participant, or both.\nSputum volume.\nMean difference in other objective indices (airway markers of inflammation, exhaled nitric oxide, etc).\nMean difference in quality of life in

## Cost estimate (no API call)

The 3.06 chars/token ratio overestimated the earlier runs by about 15-18%.

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")
est

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 7871 | max: 22893
estimated input tokens: 108.0M | estimated cost: $4.54
subset (10 reviews): $0.27


,records,chars,tokens_M,usd
review,,,,
CD000996,281,887539,0.290,0.012
CD001261,571,2223859,0.727,0.031
CD004414,336,1690465,0.552,0.023
CD006468,3874,13869609,4.533,0.190
CD007867,943,5260161,1.719,0.072
CD009069,1757,10269470,3.356,0.141
CD009642,1922,11460433,3.745,0.157
CD010038,8867,129366839,42.277,1.776
CD010239,224,1070159,0.350,0.015


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    resp = S.call(build_body(*records.loc[0, ["review", "title", "abstract"]]))
    print(json.dumps(resp, indent=1))

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

reviews:   0%|          | 0/10 [00:00<?, ?it/s]

CD000996:   0%|          | 0/281 [00:00<?, ?it/s]

CD000996:   0%|          | 1/281 [00:00<01:44,  2.69it/s]

CD000996:   2%|▏         | 7/281 [00:00<00:15, 18.18it/s]

CD000996:   4%|▍         | 11/281 [00:00<00:12, 21.43it/s]

CD000996:   6%|▌         | 16/281 [00:00<00:10, 24.60it/s]

... (860 outputs left out)

CD012551:  99%|█████████▉| 588/591 [00:20<00:00, 25.27it/s]

CD012551: 100%|██████████| 591/591 [00:20<00:00, 21.57it/s]

reviews: 100%|██████████| 10/10 [02:05<00:00, 11.49s/it]

reviews: 100%|██████████| 10/10 [02:05<00:00, 12.59s/it]

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

All configurations are scored on the same records: the reviews that have a cache file for this run and for every compared configuration. Failed calls and refusals get the lowest score and are counted. Other configurations are read from their caches, so nothing is re-requested.

In [8]:
done = sorted(set.intersection(*({p.stem for p in path.glob("*.jsonl")} for path in COMPARE.values())))
print(len(done), "of", len(CRITERIA), "reviews in every configuration")
sub = records[records.review.isin(done)].copy()
for col, path in COMPARE.items():
    sub = S.add_scores(sub, path, col)

results = {c: S.evaluate(sub, c) for c in ("bm25", *COMPARE)}
for c in ("title_abstract", "no_results", "fulltext_criteria_rq", "fulltext_criteria"):
    results[f"{c} - baseline"] = results[c] - results["baseline"]
    results[f"{c} - baseline"]["reviews"] = results["baseline"]["reviews"]
pd.concat(results)

10 of 20 reviews in every configuration
baseline: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


title_abstract: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $3.61 | without a score (ranked last): 0
no_results: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $0.16 | without a score (ranked last): 0
fulltext_criteria_rq: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $0.23 | without a score (ranked last): 0
fulltext_criteria: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $0.22 | without a score (ranked last): 0


MAP  R@1%  R@5%  R@10%  R@20%  \
bm25                            label_abs      31.4   7.5  31.0   42.2   56.3   
                                label_content  23.9   8.6  24.3   37.8   55.3   
baseline                        label_abs      70.7  21.0  60.1   73.7   84.8   
                                label_content  77.2  31.4  79.8   88.6   94.3   
title_abstract                  label_abs      73.8  19.8  60.9   76.9   87.3   
                                label_content  84.4  34.4  81.9   92.0   96.5   
no_results                      label_abs      74.4  22.3  60.4   75.8   86.8   
                                label_content  85.4  36.0  84.7   91.8   95.9   
fulltext_criteria_rq            label_abs      72.0  20.3  59.8   73.5   83.5   
                                label_content  82.1  29.5  83.0   91.6   94.9   
fulltext_criteria               label_abs      68.2  19.7  58.4   72.1   84.1   
                                label_content  77.0  26.0  82.6   91.0   94.3   
title_abstract - baseline       label_abs       3.1  -1.2   0.8    3.2    2.5   
                                label_content   7.2   3.0   2.1    3.4    2.2   
no_results - baseline           label_abs       3.7   1.3   0.3    2.1    2.0   
                                label_content   8.2   4.6   4.9    3.2    1.6   
fulltext_criteria_rq - baseline label_abs       1.3  -0.7  -0.3   -0.2   -1.3   
                                label_content   4.9  -1.9   3.2    3.0    0.6   
fulltext_criteria - baseline    label_abs      -2.5  -1.3  -1.7   -1.6   -0.7   
                                label_content  -0.2  -5.4   2.8    2.4    0.0   

                                               R@50%  WSS@95%  TNR@95%  \
bm25                            label_abs       81.3     33.8     40.9   
                                label_content   81.3     33.5     39.9   
baseline                        label_abs       96.8     60.5     71.1   
                                label_content   99.2     79.1     88.0   
title_abstract                  label_abs       97.8     64.0     75.0   
                                label_content   99.7     83.3     92.5   
no_results                      label_abs       99.0     63.8     74.8   
                                label_content   99.1     83.0     92.0   
fulltext_criteria_rq            label_abs       97.8     62.4     73.2   
                                label_content   99.7     81.9     91.2   
fulltext_criteria               label_abs       98.1     61.2     72.1   
                                label_content   99.7     83.0     92.2   
title_abstract - baseline       label_abs        1.0      3.5      3.9   
                                label_content    0.5      4.2      4.5   
no_results - baseline           label_abs        2.2      3.3      3.7   
                                label_content   -0.1      3.9      4.0   
fulltext_criteria_rq - baseline label_abs        1.0      1.9      2.1   
                                label_content    0.5      2.8      3.2   
fulltext_criteria - baseline    label_abs        1.3      0.7      1.0   
                                label_content    0.5      3.9      4.2   

                                               WSS@100%  reviews  
bm25                            label_abs          36.5     10.0  
                                label_content      36.1      9.0  
baseline                        label_abs          59.8     10.0  
                                label_content      80.3      9.0  
title_abstract                  label_abs          62.9     10.0  
                                label_content      87.1      9.0  
no_results                      label_abs          63.1     10.0  
                                label_content      86.2      9.0  
fulltext_criteria_rq            label_abs          62.7     10.0  
                                label_content      82.7      9.0  
fulltext_criteria               label_abs          59.

In [9]:
# per-review MAP
rows = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in COMPARE:
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            rows.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per_review = pd.DataFrame(rows).T
for label in ("label_abs", "label_content") if rows else ():
    d = per_review[(label, "fulltext_criteria")] - per_review[(label, "baseline")]
    print(f"{label}: fulltext_criteria vs baseline, better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
per_review

label_abs: fulltext_criteria vs baseline, better 5 / worse 5 of 10 reviews
label_content: fulltext_criteria vs baseline, better 5 / worse 3 of 9 reviews


label_abs                                                 \
          baseline title_abstract no_results fulltext_criteria_rq   
CD000996      75.5           83.7       84.3                 80.9   
CD001261      56.1           79.0       82.4                 51.6   
CD004414      78.8           80.5       79.4                 78.2   
CD007867      52.3           57.8       58.9                 59.9   
CD010239      94.2           94.3       92.9                 95.8   
CD011140      77.5           41.3       56.0                 73.3   
CD011571      77.1           82.6       82.2                 78.1   
CD011977      68.3           86.8       84.0                 82.2   
CD012164      62.9           68.6       60.4                 55.7   
CD012551      64.1           63.6       63.2                 63.9   

                           label_content                            \
         fulltext_criteria      baseline title_abstract no_results   
CD000996              80.0          87.7          100.0      100.0   
CD001261              53.2          72.6           82.7       80.9   
CD004414              75.9          79.9           58.7       73.7   
CD007867              55.8          63.4           75.3       79.1   
CD010239              94.4          99.1           99.1      100.0   
CD011140              51.0           NaN            NaN        NaN   
CD011571              69.8          80.0           95.8       97.6   
CD011977              81.8          60.0           78.9       78.2   
CD012164              56.0          75.6           91.7       80.6   
CD012551              64.5          76.7           77.2       78.8   

                                                 
         fulltext_criteria_rq fulltext_criteria  
CD000996                 97.6              97.6  
CD001261                 78.3              75.1  
CD004414                 75.0              66.7  
CD007867                 76.5              71.2  
CD010239                 99.1              99.1  
CD011140                  NaN               NaN  
CD011571                 90.6              66.8  
CD011977                 78.3              73.5  
CD012164                 63.9              63.9  
CD012551                 79.4              79.2